# 73 - Experimento MPNet + DBSCAN

Este notebook agrupa descricoes normalizadas usando `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` e DBSCAN. Usa a base completa `2025_NCMs_filtrados_semstopwords`, com uma amostra de no maximo 20.000 descricoes por semente, igual ao notebook 72. Nao usa NCM nem rotulos humanos; ruido e tratado como resultado explicito.

Execute de cima para baixo em uma sessao limpa. Cada execucao grava relatorios em uma pasta UTC nova dentro de `reports/mpnet_dbscan/` na raiz do repositorio. O primeiro uso do MPNet pode baixar o modelo do Hugging Face.


In [1]:
from sklearn.decomposition import TruncatedSVD
from hashlib import sha256
from datetime import datetime, timezone
from pathlib import Path
import json
import logging

import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
from IPython.display import display
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from matplotlib.colors import ListedColormap
from sklearn import config_context

from nofis_classifier.schema import DESCRIPTION_NORMALIZED, FREQUENCY

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("notebook73")
pd.set_option("display.max_colwidth", 140)


## Configuracoes e comparabilidade

Controles comuns aos notebooks 70 a 76:

- Base completa `2025_NCMs_filtrados_semstopwords`, periodo 202501-202512.
- Ate 20.000 descricoes unicas para agrupamento, amostradas com semente 42 e ordenadas por descricao normalizada. Frequencia nao pondera o ajuste.
- Silhouette cosseno sobre uma amostra base fixa de ate 20.000 linhas, com semente 42. Ruido do algoritmo e excluido depois da amostragem; a quantidade efetivamente avaliada e registrada.
- Ranking exige silhouette definida e de 10 a 1.000 clusters efetivamente formados, sem contar ruido. Pesos: 0,75 para silhouette reescalada e 0,25 para cobertura do algoritmo.
- t-SNE 2D e 3D: mesmos ate 5.000 itens, semente 42, TruncatedSVD de ate 50 componentes, perplexidade 30, 1.000 iteracoes, init PCA (quando ha dimensoes suficientes), learning_rate auto e Barnes-Hut. As projecoes de representacoes diferentes sao exploratorias, nao um espaco comum de distancias.
- `comparison` em `run.json` registra os controles e o hash das descricoes/frequencias usadas no agrupamento.

MPNet: `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`, batch_size 64, normalize_embeddings True e saida float32. Nao ha min_df ou analyzer no MPNet puro.

DBSCAN: a mesma grade de eps e min_samples do notebook 72. Grade atual: 11 valores de eps x 8 valores de min_samples = 88 ajustes.


In [2]:
DATASET_NAME = "2025_NCMs_filtrados_semstopwords"

PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data" / "processed").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Nao encontrei a raiz do repositorio com data/processed/.")
DATASET_DIR = PROJECT_ROOT / "data" / "processed" / DATASET_NAME
DATASET_FILES = sorted(DATASET_DIR.glob("*.parquet"))
if not DATASET_FILES:
    raise FileNotFoundError(f"Nenhum Parquet encontrado para {DATASET_NAME}: {DATASET_DIR}")
if len(DATASET_FILES) > 1:
    raise ValueError(
        f"Mais de um Parquet encontrado em {DATASET_DIR}; selecione o arquivo/periodo explicitamente: "
        f"{[path.name for path in DATASET_FILES]}"
    )
DATASET_PATH = DATASET_FILES[0]
if not DATASET_PATH.stem.startswith(f"{DATASET_NAME}-"):
    raise ValueError(
        f"O nome do arquivo deve comecar com '{DATASET_NAME}-' para derivar o periodo: "
        f"{DATASET_PATH.name}"
    )
PERIOD_LABEL = DATASET_PATH.stem[len(DATASET_NAME) + 1:]
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
MAX_DESCRIPTIONS = 20_000
CONSIDER_FREQUENCY_IN_CLUSTERING = False
SAMPLE_SEEDS = [42]
ENCODE_BATCH_SIZE = 64
SILHOUETTE_SAMPLE_SIZE = 20_000
DBSCAN_N_JOBS = 1
PAIRWISE_WORKING_MEMORY_MIB = 64

EPS_VALUES = [0.01, 0.02, 0.025, 0.03, 0.035, 0.04, 0.05, 0.07, 0.1, 0.5, 1]
MIN_SAMPLES_VALUES = [2, 3, 5, 7, 10, 20, 30, 100]
OUTPUT_ROOT = PROJECT_ROOT / Path("reports/mpnet_dbscan")


MAX_CLUSTERS_FOR_RANKING = 1000
MIN_CLUSTER_FOR_RANKING = 10
SILHOUETTE_WEIGHT = 0.75
NOISE_WEIGHT = 0.25

TSNE_MAX_POINTS = 5_000
TSNE_PERPLEXITY = 30
TSNE_MAX_ITER = 1_000
TSNE_RANDOM_STATE = 42
METRIC_RANDOM_STATE = 42

TSNE_SVD_COMPONENTS = 50

TSNE_3D_N_COMPONENTS = 3


## 2. Fun??es da an?lise

A prepara??o consolida duplicatas sem alterar a frequ?ncia total. O resumo mant?m o r?tulo `-1` para ru?do. A silhouette exclui ru?do e s? ? calculada quando h? pelo menos dois clusters e mais de um item em cada parti??o considerada.


In [3]:
def prepare_items(frame):
    """Limpa descri??es, agrega duplicatas e preserva a frequ?ncia total."""
    initial_frequency = frame[FREQUENCY].sum()
    prepared = frame[[DESCRIPTION_NORMALIZED, FREQUENCY]].copy()
    prepared[DESCRIPTION_NORMALIZED] = prepared[DESCRIPTION_NORMALIZED].astype("string").str.strip()
    prepared = prepared[prepared[DESCRIPTION_NORMALIZED].notna() & prepared[DESCRIPTION_NORMALIZED].ne("")]
    prepared = prepared[prepared[FREQUENCY] >= 1].copy()
    prepared = prepared.groupby(DESCRIPTION_NORMALIZED, as_index=False, sort=False)[FREQUENCY].sum()
    if prepared[FREQUENCY].sum() != initial_frequency:
        raise ValueError("A prepara??o alterou a soma de frequ?ncia.")
    return prepared.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)


def score_configuration(matrix, labels, sample_size, random_state):
    """Avalia a mesma amostra de linhas, excluindo ruido somente apos amostrar."""
    labels = np.asarray(labels)
    noise_mask = labels == -1
    cluster_count = len(np.unique(labels[~noise_mask]))
    base_count = min(sample_size, len(labels))
    rng = np.random.default_rng(random_state)
    positions = np.sort(rng.choice(len(labels), size=base_count, replace=False))
    positions = positions[~noise_mask[positions]]
    sample_labels = labels[positions]
    sample_cluster_count = len(np.unique(sample_labels))
    score = np.nan
    actual_count = len(positions)
    if 2 <= sample_cluster_count < actual_count:
        with config_context(working_memory=PAIRWISE_WORKING_MEMORY_MIB):
            score = float(silhouette_score(matrix[positions], sample_labels, metric="cosine"))
    else:
        actual_count = 0
    return {
        "cluster_count_excluding_noise": cluster_count,
        "noise_count": int(noise_mask.sum()),
        "noise_fraction": float(noise_mask.mean()),
        "silhouette_base_sample_size": base_count,
        "silhouette_sample_size": actual_count,
        "silhouette_cosine_without_noise": score,
    }


def summarize_clusters(items, labels):
    """Retorna atribui??es e resumo dos clusters, incluindo a linha de ru?do."""
    result = items.copy()
    result["cluster"] = labels
    result["ruido"] = result["cluster"].eq(-1)
    summary = (
        result.groupby("cluster", as_index=False)
        .agg(descricoes_unicas=(DESCRIPTION_NORMALIZED, "size"), frequencia_total=(FREQUENCY, "sum"))
        .sort_values("descricoes_unicas", ascending=False)
    )
    return result, summary



def compute_tsne_projection(features, max_points, perplexity, max_iter, random_state, n_components=2):
    """Usa a mesma amostra, SVD e parametros t-SNE em todas as representacoes."""
    point_count = features.shape[0]
    sample_count = min(point_count, max_points)
    if sample_count < 3 or n_components not in (2, 3):
        raise ValueError("t-SNE exige pelo menos tres itens e n_components 2 ou 3.")
    rng = np.random.default_rng(random_state)
    indices = np.sort(rng.choice(point_count, size=sample_count, replace=False))
    sampled = features[indices]
    dimensions = min(TSNE_SVD_COMPONENTS, sampled.shape[1] - 1, sample_count - 1)
    if dimensions >= 1:
        reduced = TruncatedSVD(n_components=dimensions, random_state=random_state).fit_transform(sampled)
    else:
        reduced = sampled.toarray() if hasattr(sampled, "toarray") else np.asarray(sampled)
        dimensions = reduced.shape[1]
    coordinates = TSNE(
        n_components=n_components,
        perplexity=min(float(perplexity), (sample_count - 1) / 3),
        init="pca" if dimensions >= n_components else "random",
        learning_rate="auto", max_iter=max_iter, random_state=random_state,
        method="barnes_hut",
    ).fit_transform(reduced)
    return indices, coordinates, dimensions


def save_tsne_plot(coordinates, labels, run_id, output_path):
    """Salva um PNG t-SNE colorido pelos clusters, com o ruido em cinza."""
    labels = np.asarray(labels)
    noise_mask = labels == -1
    cluster_ids = np.unique(labels[~noise_mask])
    if cluster_ids.size:
        palette = plt.get_cmap("tab20").colors
        cluster_cmap = ListedColormap([
            palette[index % len(palette)] for index in range(len(cluster_ids))
        ])
        cluster_codes = np.searchsorted(cluster_ids, labels[~noise_mask])
    else:
        cluster_cmap = None
        cluster_codes = np.empty(0, dtype=int)

    figure, axis = plt.subplots(figsize=(11, 8), constrained_layout=True)
    if noise_mask.any():
        axis.scatter(
            coordinates[noise_mask, 0], coordinates[noise_mask, 1],
            c="#b8b8b8", s=8, alpha=0.3, linewidths=0,
        )
    if cluster_ids.size:
        axis.scatter(
            coordinates[~noise_mask, 0], coordinates[~noise_mask, 1],
            c=cluster_codes, cmap=cluster_cmap, vmin=-0.5,
            vmax=len(cluster_ids) - 0.5, s=9, alpha=0.75, linewidths=0,
        )
    axis.set_title(
        f"t-SNE: {run_id}\n{len(cluster_ids)} clusters; "
        f"{int(noise_mask.sum()):,} descricoes como ruido"
    )
    axis.set_xlabel("t-SNE 1")
    axis.set_ylabel("t-SNE 2")
    axis.text(
        0.01, 0.01,
        "Ruido em cinza; as cores se repetem a cada 20 clusters.",
        transform=axis.transAxes, fontsize=8, color="#555555",
    )
    figure.savefig(output_path, dpi=160)
    plt.close(figure)


def save_tsne_3d_html(coordinates, labels, run_id, output_path):
    """Salva uma visualiza??o t-SNE 3D interativa e autocontida em HTML."""
    points = [
        {"x": float(x), "y": float(y), "z": float(z), "label": int(label)}
        for (x, y, z), label in zip(coordinates, labels)
    ]
    data_json = json.dumps(points, separators=(",", ":"))
    title = str(run_id).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    page = r"""<!doctype html>
<html lang="pt-BR">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>t-SNE 3D: __TITLE__</title>
<style>
html, body { margin: 0; height: 100%; background: #f7f8fa; color: #202124; font: 14px system-ui, sans-serif; }
main { display: flex; flex-direction: column; height: 100%; }
header { padding: 12px 18px; background: white; border-bottom: 1px solid #ddd; }
h1 { margin: 0 0 4px; font-size: 17px; }
p { margin: 0; color: #555; }
canvas { display: block; width: 100%; flex: 1; min-height: 300px; cursor: grab; }
canvas:active { cursor: grabbing; }
</style>
</head>
<body>
<main>
<header><h1>Proje??o t-SNE 3D: __TITLE__</h1><p>Arraste para girar; use a roda do mouse para aproximar. Ru?do em cinza.</p></header>
<canvas id="plot"></canvas>
</main>
<script>
const points = __POINTS__;
const canvas = document.getElementById("plot");
const ctx = canvas.getContext("2d");
let yaw = 0.65, pitch = -0.35, zoom = 1, dragging = false, lastX = 0, lastY = 0;
const colors = ["#3366cc", "#dc3912", "#ff9900", "#109618", "#990099", "#0099c6", "#dd4477", "#66aa00", "#b82e2e", "#316395", "#994499", "#22aa99", "#aaaa11", "#6633cc", "#e67300", "#8b0707", "#651067", "#329262", "#5574a6", "#3b3eac"];
const bounds = ["x", "y", "z"].map(key => {
  const values = points.map(point => point[key]);
  const low = Math.min(...values), high = Math.max(...values);
  return { low, span: Math.max(high - low, 1e-9) };
});
function resize() {
  const ratio = window.devicePixelRatio || 1;
  canvas.width = Math.max(1, Math.floor(canvas.clientWidth * ratio));
  canvas.height = Math.max(1, Math.floor(canvas.clientHeight * ratio));
  ctx.setTransform(ratio, 0, 0, ratio, 0, 0);
  draw();
}
function draw() {
  const width = canvas.clientWidth, height = canvas.clientHeight;
  ctx.clearRect(0, 0, width, height);
  const scale = Math.min(width, height) * 0.7 * zoom;
  const cy = Math.cos(yaw), sy = Math.sin(yaw), cp = Math.cos(pitch), sp = Math.sin(pitch);
  const projected = points.map(point => {
    const x = ((point.x - bounds[0].low) / bounds[0].span - 0.5) * 2;
    const y = ((point.y - bounds[1].low) / bounds[1].span - 0.5) * 2;
    const z = ((point.z - bounds[2].low) / bounds[2].span - 0.5) * 2;
    const rx = x * cy - z * sy, rz = x * sy + z * cy;
    const ry = y * cp - rz * sp, depth = y * sp + rz * cp;
    return { x: width / 2 + rx * scale / 2, y: height / 2 - ry * scale / 2, depth, label: point.label };
  }).sort((a, b) => a.depth - b.depth);
  for (const point of projected) {
    ctx.globalAlpha = point.label === -1 ? 0.24 : 0.78;
    ctx.fillStyle = point.label === -1 ? "#999999" : colors[Math.abs(point.label) % colors.length];
    ctx.beginPath();
    ctx.arc(point.x, point.y, point.label === -1 ? 2 : 3, 0, Math.PI * 2);
    ctx.fill();
  }
  ctx.globalAlpha = 1;
}
canvas.addEventListener("pointerdown", event => { dragging = true; lastX = event.clientX; lastY = event.clientY; canvas.setPointerCapture(event.pointerId); });
canvas.addEventListener("pointerup", () => { dragging = false; });
canvas.addEventListener("pointermove", event => {
  if (!dragging) return;
  yaw += (event.clientX - lastX) * 0.008;
  pitch = Math.max(-1.5, Math.min(1.5, pitch + (event.clientY - lastY) * 0.008));
  lastX = event.clientX; lastY = event.clientY; draw();
});
canvas.addEventListener("wheel", event => { event.preventDefault(); zoom = Math.max(0.3, Math.min(5, zoom * (event.deltaY < 0 ? 1.1 : 0.9))); draw(); }, { passive: false });
window.addEventListener("resize", resize);
resize();
</script>
</body>
</html>"""
    page = page.replace("__TITLE__", title).replace("__POINTS__", data_json)
    output_path.write_text(page, encoding="utf-8")

def comparison_metadata(frame):
    """Registra controles comuns e a identidade exata da amostra de agrupamento."""
    sample = frame[[DESCRIPTION_NORMALIZED, FREQUENCY]].sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)
    return {
        "dataset_name": DATASET_NAME,
        "training_description_count": len(sample),
        "training_sample_sha256": sha256(pd.util.hash_pandas_object(sample, index=False).values.tobytes()).hexdigest(),
        "max_descriptions": MAX_DESCRIPTIONS, "sample_seed": 42,
        "frequency_weighted": False,
        "silhouette_metric": "cosine", "silhouette_sample_size": SILHOUETTE_SAMPLE_SIZE,
        "metric_random_state": METRIC_RANDOM_STATE,
        "metric_sampling": "fixed rows, then exclude algorithmic noise",
        "min_cluster_for_ranking": MIN_CLUSTER_FOR_RANKING,
        "max_clusters_for_ranking": MAX_CLUSTERS_FOR_RANKING,
        "silhouette_weight": SILHOUETTE_WEIGHT, "noise_weight": NOISE_WEIGHT,
        "tsne": {
            "max_points": TSNE_MAX_POINTS, "perplexity": TSNE_PERPLEXITY,
            "max_iter": TSNE_MAX_ITER, "random_state": TSNE_RANDOM_STATE,
            "reduction": "TruncatedSVD", "svd_components": TSNE_SVD_COMPONENTS,
            "dimensions": [2, 3], "init": "pca", "learning_rate": "auto",
            "method": "barnes_hut",
        },
    }


## 3. Carregar, preparar e gerar embeddings

A amostra e selecionada separadamente para cada semente. O modelo MPNet transforma cada amostra em embeddings normalizados, que serao reutilizados em toda a grade de DBSCAN daquela semente.


In [4]:
logger.info("Base selecionada: %s", DATASET_PATH)
raw_items = pd.read_parquet(DATASET_PATH)
required_columns = {DESCRIPTION_NORMALIZED, FREQUENCY}
missing_columns = required_columns - set(raw_items.columns)
if missing_columns:
    raise ValueError(f"Colunas ausentes no dataset: {sorted(missing_columns)}")
base_items = prepare_items(raw_items)
logger.info("Descricoes disponiveis apos preparacao: %d", len(base_items))
logger.info("Carregando modelo %s", MODEL_NAME)
encoder = SentenceTransformer(MODEL_NAME)

items_by_seed = {}
vectors_by_seed = {}
tsne_by_seed = {}
tsne_3d_by_seed = {}
for sample_seed in SAMPLE_SEEDS:
    sampled_items = base_items
    if MAX_DESCRIPTIONS is not None and len(base_items) > MAX_DESCRIPTIONS:
        sampled_items = base_items.sample(n=MAX_DESCRIPTIONS, random_state=sample_seed)
    sampled_items = sampled_items.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)
    vectors = encoder.encode(
        sampled_items[DESCRIPTION_NORMALIZED].astype(str).tolist(),
        batch_size=ENCODE_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)
    logger.info("Calculando t-SNE para seed=%d", sample_seed)
    tsne_indices, tsne_coordinates, _ = compute_tsne_projection(
        vectors, TSNE_MAX_POINTS, TSNE_PERPLEXITY, TSNE_MAX_ITER,
        TSNE_RANDOM_STATE,
    )
    tsne_3d_indices, tsne_3d_coordinates, _ = compute_tsne_projection(
        vectors, TSNE_MAX_POINTS, TSNE_PERPLEXITY, TSNE_MAX_ITER,
        TSNE_RANDOM_STATE, n_components=TSNE_3D_N_COMPONENTS,
    )
    assert np.array_equal(tsne_indices, tsne_3d_indices)
    tsne_3d_by_seed[sample_seed] = {"indices": tsne_3d_indices, "coordinates": tsne_3d_coordinates}
    tsne_by_seed[sample_seed] = {"indices": tsne_indices, "coordinates": tsne_coordinates}
    items_by_seed[sample_seed] = sampled_items
    vectors_by_seed[sample_seed] = vectors
    logger.info(
        "Seed %d: %d descricoes, embeddings %s, frequencia total %d",
        sample_seed, len(sampled_items), vectors.shape, sampled_items[FREQUENCY].sum(),
    )

display(pd.DataFrame([
    {
        "sample_seed": sample_seed,
        "descricoes": len(items_by_seed[sample_seed]),
        "dimensoes_embedding": vectors_by_seed[sample_seed].shape[1],
        "frequencia_total": int(items_by_seed[sample_seed][FREQUENCY].sum()),
    }
    for sample_seed in SAMPLE_SEEDS
]))


2026-10-25 00:56:08,561 INFO Base selecionada: C:\Users\samuel.simao\Sources\Nofis-Classifier\data\processed\2025_NCMs_filtrados_semstopwords\2025_NCMs_filtrados_semstopwords-202501-202512.parquet


2026-10-25 00:56:08,710 INFO Descricoes disponiveis apos preparacao: 146999


2026-10-25 00:56:08,711 INFO Carregando modelo sentence-transformers/paraphrase-multilingual-mpnet-base-v2


2026-10-25 00:56:08,713 INFO No device provided, using cpu


2026-10-25 00:56:08,737 INFO Loading SentenceTransformer model from sentence-transformers/paraphrase-multilingual-mpnet-base-v2.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

2026-10-25 00:59:42,709 INFO Calculando t-SNE para seed=42


2026-10-25 01:00:21,369 INFO Seed 42: 20000 descricoes, embeddings (20000, 768), frequencia total 117883


,sample_seed,descricoes,dimensoes_embedding,frequencia_total
0,42,20000,768,117883


## 4. Ajustar DBSCAN e comparar parametros

DBSCAN usa distancia cosseno nos vetores densos do MPNet. `CONSIDER_FREQUENCY_IN_CLUSTERING` controla o `sample_weight`; por padrao, cada descricao unica conta uma vez. Silhouette e fracao de ruido tambem contam descricoes unicas. Os embeddings de cada semente sao reutilizados nos ajustes correspondentes.

A projecao t-SNE e calculada uma vez por semente sobre uma amostra fixa; cada combinacao de `eps` e `min_samples` reutiliza as mesmas coordenadas no PNG.


In [5]:
if not 1 <= MIN_CLUSTER_FOR_RANKING <= MAX_CLUSTERS_FOR_RANKING:
    raise ValueError(
        "Os limites devem atender a 1 <= MIN_CLUSTER_FOR_RANKING <= MAX_CLUSTERS_FOR_RANKING."
    )

logger.info("Grade planejada: %d ajustes", len(SAMPLE_SEEDS) * len(EPS_VALUES) * len(MIN_SAMPLES_VALUES))

records = []
labels_by_run = {}
items_by_run = {}
for sample_seed in SAMPLE_SEEDS:
    items = items_by_seed[sample_seed]
    vectors = vectors_by_seed[sample_seed]
    for eps in EPS_VALUES:
        for min_samples in MIN_SAMPLES_VALUES:
            experiment_id = f"seed{sample_seed}_eps{eps:g}_min_samples{min_samples}"
            logger.info("Ajustando DBSCAN %s", experiment_id)
            model = DBSCAN(
                eps=eps, min_samples=min_samples, metric="cosine",
                algorithm="brute", n_jobs=DBSCAN_N_JOBS,
            )
            with config_context(working_memory=PAIRWISE_WORKING_MEMORY_MIB):
                sample_weight = (
                    items[FREQUENCY].to_numpy(dtype=float)
                    if CONSIDER_FREQUENCY_IN_CLUSTERING else None
                )
                model.fit(vectors, sample_weight=sample_weight)
            labels = model.labels_
            with config_context(working_memory=PAIRWISE_WORKING_MEMORY_MIB):
                metrics = score_configuration(
                    vectors, labels, SILHOUETTE_SAMPLE_SIZE,
                    METRIC_RANDOM_STATE,
                )
            records.append({
                "run_id": experiment_id, "sample_seed": sample_seed,
                "eps": eps, "min_samples": min_samples,
                "frequency_weighted": CONSIDER_FREQUENCY_IN_CLUSTERING, **metrics,
            })
            labels_by_run[experiment_id] = labels
            items_by_run[experiment_id] = items
results = pd.DataFrame(records)

if SILHOUETTE_WEIGHT < 0 or NOISE_WEIGHT < 0 or SILHOUETTE_WEIGHT + NOISE_WEIGHT <= 0:
    raise ValueError("Os pesos devem ser nao negativos e pelo menos um deles deve ser maior que zero.")

# Silhouette varia de -1 a 1; reescale para 0 a 1 para combinar com cobertura.
results["silhouette_quality"] = (results["silhouette_cosine_without_noise"] + 1) / 2
results["noise_quality"] = 1 - results["noise_fraction"]
weight_total = SILHOUETTE_WEIGHT + NOISE_WEIGHT
results["composite_score"] = (
    SILHOUETTE_WEIGHT * results["silhouette_quality"]
    + NOISE_WEIGHT * results["noise_quality"]
) / weight_total

# O score composto e o criterio principal; os outros criterios so desempataram scores iguais.
ranking = (
    results.loc[
        results["cluster_count_excluding_noise"].between(
            MIN_CLUSTER_FOR_RANKING, MAX_CLUSTERS_FOR_RANKING, inclusive="both",
        ) & results["silhouette_cosine_without_noise"].notna()
    ]
    .sort_values(
        ["composite_score", "silhouette_cosine_without_noise", "noise_fraction"],
        ascending=[False, False, True], na_position="last",
    )
    .reset_index(drop=True)
)
if ranking.empty:
    raise ValueError(
        f"Nenhum resultado atende ao intervalo de {MIN_CLUSTER_FOR_RANKING} a "
        f"{MAX_CLUSTERS_FOR_RANKING} clusters. Ajuste eps/min_samples ou o limite."
    )
# Explicita a posicao de cada configuracao depois dos filtros e criterios acima.
ranking.insert(0, "rank", range(1, len(ranking) + 1))

display(ranking)


2026-10-25 01:00:21,432 INFO Grade planejada: 88 ajustes


2026-10-25 01:00:21,437 INFO Ajustando DBSCAN seed42_eps0.01_min_samples2


2026-10-25 01:00:29,016 INFO Ajustando DBSCAN seed42_eps0.01_min_samples3


2026-10-25 01:00:35,752 INFO Ajustando DBSCAN seed42_eps0.01_min_samples5


2026-10-25 01:00:43,258 INFO Ajustando DBSCAN seed42_eps0.01_min_samples7


...

2026-10-25 01:10:05,178 INFO Ajustando DBSCAN seed42_eps1_min_samples10


2026-10-25 01:10:15,739 INFO Ajustando DBSCAN seed42_eps1_min_samples20


2026-10-25 01:10:25,873 INFO Ajustando DBSCAN seed42_eps1_min_samples30


2026-10-25 01:10:35,687 INFO Ajustando DBSCAN seed42_eps1_min_samples100


,rank,run_id,sample_seed,eps,min_samples,frequency_weighted,cluster_count_excluding_noise,noise_count,noise_fraction,silhouette_base_sample_size,silhouette_sample_size,silhouette_cosine_without_noise,silhouette_quality,noise_quality,composite_score
0,1,seed42_eps0.01_min_samples5,42,0.010,5,False,11,19867,0.99335,20000,133,0.975304,0.987652,0.00665,0.742401
1,2,seed42_eps0.02_min_samples7,42,0.020,7,False,10,19859,0.99295,20000,141,0.964061,0.982030,0.00705,0.738285
2,3,seed42_eps0.02_min_samples5,42,0.020,5,False,17,19822,0.99110,20000,178,0.961619,0.980810,0.00890,0.737832
3,4,seed42_eps0.025_min_samples7,42,0.025,7,False,11,19849,0.99245,20000,151,0.961143,0.980571,0.00755,0.737316
4,5,seed42_eps0.03_min_samples7,42,0.030,7,False,12,19841,0.99205,20000,159,0.958833,0.979416,0.00795,0.736550
5,6,seed42_eps0.025_min_samples5,42,0.025,5,False,28,19762,0.98810,20000,238,0.943481,0.971741,0.01190,0.731780
6,7,seed42_eps0.01_min_samples2,42,0.010,2,False,189,19486,0.97430,20000,514,0.916927,0.958463,0.02570,0.725272
7,8,seed42_eps0.02_min_samples2,42,0.020,2,False,469,18825,0.94125,20000,1175,0.876395,0.938198,0.05875,0.718336
8,9,seed42_eps0.02_min_samples3,42,0.020,3,False,88,19587,0.97935,20000,413,0.901033,0.950517,0.02065,0.718050
9,10,seed42_eps0.025_min_samples2,42,0.025,2,False,635,18380,0.91900,20000,1620,0.830410,0.915205,0.08100,0.706654


## 5. Revisar um resultado e salvar relatorios

A selecao automatica usa o primeiro resultado do ranking composto, apos o filtro de no maximo 200 clusters. Revise quantidade de clusters, silhouette e cobertura antes de interpretar. `metrics.csv` preserva toda a grade e `ranking.csv` registra os resultados ordenados apos o filtro.

Os assignments e o resumo correspondem ao `SELECTED_RUN_ID`. Cada execucao grava uma pasta UTC nova, preservando resultados anteriores. Os PNG ficam em `tsne_plots/` e as coordenadas da amostra em `tsne_coordinates/`.


In [6]:
SELECTED_RUN_ID = ranking.iloc[0]["run_id"]
selected = results.set_index("run_id").loc[SELECTED_RUN_ID]
selected_items = items_by_run[SELECTED_RUN_ID]
selected_labels = labels_by_run[SELECTED_RUN_ID]
assignments, cluster_summary = summarize_clusters(selected_items, selected_labels)
cluster_summary = cluster_summary.sort_values("frequencia_total", ascending=False).reset_index(drop=True)
cluster_sizes = assignments.loc[assignments["cluster"].ne(-1)].groupby("cluster")[DESCRIPTION_NORMALIZED].size()
singleton_clusters = cluster_sizes[cluster_sizes.eq(1)].index
frequent_singletons = assignments[assignments["cluster"].isin(singleton_clusters)].sort_values(
    FREQUENCY, ascending=False
)
frequent_noise = assignments[assignments["cluster"].eq(-1)].sort_values(
    FREQUENCY, ascending=False
)
display(cluster_summary.head(20))
display(frequent_singletons.head(20))
display(frequent_noise.head(20))

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ%f")
output_dir = OUTPUT_ROOT / f"{PERIOD_LABEL}_{RUN_ID}"
output_dir.mkdir(parents=True, exist_ok=False)
results.to_csv(output_dir / "metrics.csv", index=False)
ranking.to_csv(output_dir / "ranking.csv", index=False)
assignments.to_csv(output_dir / "assignments.csv", index=False, encoding="utf-8-sig")
cluster_summary.to_csv(output_dir / "cluster_summary.csv", index=False, encoding="utf-8-sig")
frequent_singletons.to_csv(
    output_dir / "frequent_singleton_descriptions.csv", index=False, encoding="utf-8-sig"
)
frequent_noise.to_csv(
    output_dir / "frequent_noise_descriptions.csv", index=False, encoding="utf-8-sig"
)

# Uma projecao por amostra e reutilizada em todas as combinacoes de eps/min_samples.
tsne_coordinates_dir = output_dir / "tsne_coordinates"
tsne_plots_dir = output_dir / "tsne_plots"
tsne_coordinates_dir.mkdir()
tsne_plots_dir.mkdir()
tsne_3d_html_dir = output_dir / "tsne_3d_html"
tsne_3d_html_dir.mkdir()
tsne_artifacts = {}
for sample_seed, projection in tsne_by_seed.items():
    indices = projection["indices"]
    coordinates = projection["coordinates"]
    coordinate_frame = items_by_seed[sample_seed].iloc[indices][
        [DESCRIPTION_NORMALIZED, FREQUENCY]
    ].reset_index(drop=True)
    coordinate_frame["tsne_1"] = coordinates[:, 0]
    coordinate_frame["tsne_2"] = coordinates[:, 1]
    coordinate_name = f"seed{sample_seed}_coordinates.csv"
    coordinate_frame.to_csv(
        tsne_coordinates_dir / coordinate_name, index=False, encoding="utf-8-sig",
    )
    tsne_artifacts[f"seed{sample_seed}"] = {
        "coordinates_file": f"tsne_coordinates/{coordinate_name}",
        "sample_size": len(indices),
    }

for run_id, labels in labels_by_run.items():
    sample_seed = int(results.loc[results["run_id"].eq(run_id), "sample_seed"].iloc[0])
    projection = tsne_by_seed[sample_seed]
    plot_labels = np.asarray(labels)[projection["indices"]]
    plot_path = tsne_plots_dir / f"{run_id}.png"
    save_tsne_plot(projection["coordinates"], plot_labels, run_id, plot_path)
    html_path = tsne_3d_html_dir / f"{run_id}.html"
    save_tsne_3d_html(tsne_3d_by_seed[sample_seed]["coordinates"], plot_labels, run_id, html_path)
    tsne_artifacts[f"plot_3d:{run_id}"] = str(html_path.relative_to(output_dir))
    tsne_artifacts[f"plot:{run_id}"] = str(plot_path.relative_to(output_dir))

logger.info("Imagens t-SNE salvas: %d", len(labels_by_run))
run_metadata = {
    "comparison": comparison_metadata(items_by_seed[SAMPLE_SEEDS[0]]),
    "dataset_name": DATASET_NAME, "period": PERIOD_LABEL,
    "dataset_path": str(DATASET_PATH), "max_descriptions": MAX_DESCRIPTIONS,
    "sample_seeds": SAMPLE_SEEDS, "model_name": MODEL_NAME,
    "embedding_normalized": True,
    "frequency_weighted": CONSIDER_FREQUENCY_IN_CLUSTERING,
    "selected_run_id": SELECTED_RUN_ID, "eps_values": EPS_VALUES,
    "min_samples_values": MIN_SAMPLES_VALUES,
    "silhouette_sample_size": SILHOUETTE_SAMPLE_SIZE,
    "pairwise_working_memory_mib": PAIRWISE_WORKING_MEMORY_MIB,
    "max_clusters_for_ranking": MAX_CLUSTERS_FOR_RANKING,
    "min_cluster_for_ranking": MIN_CLUSTER_FOR_RANKING,
    "silhouette_weight": SILHOUETTE_WEIGHT, "noise_weight": NOISE_WEIGHT,
    "tsne": {
        "max_points": TSNE_MAX_POINTS, "perplexity": TSNE_PERPLEXITY,
        "max_iter": TSNE_MAX_ITER, "random_state": TSNE_RANDOM_STATE,
        "artifacts": tsne_artifacts,
    },
}
(output_dir / "run.json").write_text(json.dumps(run_metadata, indent=2, ensure_ascii=False), encoding="utf-8")
logger.info("Relatorios e imagens t-SNE salvos em %s", output_dir)


,cluster,descricoes_unicas,frequencia_total
0,-1,19867,117624
1,9,60,95
2,7,8,33
3,10,7,29
4,4,7,22
5,2,7,20
6,0,6,15
7,5,11,14
8,3,9,11
9,1,8,9


,descricao_normalizada,frequencia,cluster,ruido


,descricao_normalizada,frequencia,cluster,ruido
15013,pilha alcalina go tech aaa lr03 palito 2un,3588,-1,True
14987,pilha alcalina aa c 2un,1917,-1,True
10531,jantar,1493,-1,True
12606,milho verde,1464,-1,True
18043,salsicha,1339,-1,True
13954,pao forma,908,-1,True
14834,peito peru,907,-1,True
18070,salsicha kg,906,-1,True
13101,multiuso 500ml ye advir,871,-1,True
5507,cola bastao 40gr ref 4545,859,-1,True


2026-10-25 01:11:00,646 INFO Imagens t-SNE salvas: 88


2026-10-25 01:11:00,668 INFO Relatorios e imagens t-SNE salvos em C:\Users\samuel.simao\Sources\Nofis-Classifier\reports\mpnet_dbscan\202501-202512_20261025T041045Z274853


### Interpreta??o

- `noise_fraction` alta indica que muitas descri??es ficaram fora de regi?es densas para os par?metros escolhidos.
- Silhouette deve ser interpretada junto da quantidade de clusters e da fra??o de ru?do; uma pontua??o alta pode refletir poucos pontos classificados.
- A compara??o com TF-IDF ? explorat?ria: os dois m?todos usam a mesma amostra, mas representam similaridade de formas diferentes.
- DBSCAN n?o fornece uma opera??o direta para atribuir novas descri??es a clusters ap?s o ajuste e n?o define categorias sem revis?o humana.
